# Corporate Financial Performance Analysis with PySpark
Analyze annual financial statements from SEC filings submitted in 2025. Compare profitability and financial health across SIC industries. Filing year is not necessarily fiscal year.

## 1. Spark setup

In [1]:
# Start a local Spark session using available CPU cores.
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pathlib import Path

spark = (
    SparkSession.builder
    .appName("Corporate Financial Performance Analysis")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

print("Spark version:", spark.version)

26/11/24 17:28:27 WARN Utils: Your hostname, Tiens-MacBook-M4-Pro.local resolves to a loopback address: 127.0.0.1; using 10.198.168.68 instead (on interface en0)
26/11/24 17:28:27 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/24 17:28:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 3.5.5


26/11/24 17:28:27 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
# Locate the Spark UI for monitoring jobs and SQL execution.
print(spark.sparkContext.uiWebUrl)

http://10.198.168.68:4041


## 2. Load SEC data

In [3]:
# Resolve the project root and load all quarterly tab-separated files.
ROOT = Path.cwd()

if ROOT.name == "notebooks":
    ROOT = ROOT.parent

RAW = ROOT / "data" / "raw"

num = (
    spark.read
    .option("header", True)
    .option("sep", "\t")
    .csv(str(RAW / "*" / "num.txt"))
)

sub = (
    spark.read
    .option("header", True)
    .option("sep", "\t")
    .csv(str(RAW / "*" / "sub.txt"))
)

## 3. Explore the data

In [4]:
# Inspect column names and data types; CSV fields default to strings.
num.printSchema()
sub.printSchema()

root
 |-- adsh: string (nullable = true)
 |-- tag: string (nullable = true)
 |-- version: string (nullable = true)
 |-- ddate: string (nullable = true)
 |-- qtrs: string (nullable = true)
 |-- uom: string (nullable = true)
 |-- segments: string (nullable = true)
 |-- coreg: string (nullable = true)
 |-- value: string (nullable = true)
 |-- footnote: string (nullable = true)

root
 |-- adsh: string (nullable = true)
 |-- cik: string (nullable = true)
 |-- name: string (nullable = true)
 |-- sic: string (nullable = true)
 |-- countryba: string (nullable = true)
 |-- stprba: string (nullable = true)
 |-- cityba: string (nullable = true)
 |-- zipba: string (nullable = true)
 |-- bas1: string (nullable = true)
 |-- bas2: string (nullable = true)
 |-- baph: string (nullable = true)
 |-- countryma: string (nullable = true)
 |-- stprma: string (nullable = true)
 |-- cityma: string (nullable = true)
 |-- zipma: string (nullable = true)
 |-- mas1: string (nullable = true)
 |-- mas2: string (null

In [5]:
# Trigger actions to count records in both datasets.
print("Financial records:", f"{num.count():,}")
print("Filing records:", f"{sub.count():,}")

Financial records: 14,621,539
Filing records: 26,085


In [6]:
# Inspect selected numerical financial facts.
num.select(
    "adsh",
    "tag",
    "ddate",
    "qtrs",
    "uom",
    "value",
    "segments",
    "coreg"
).show(10, truncate=False)

+--------------------+--------------------------------------------------------------------+--------+----+---+----------------+----------------------------------------------------------------------------+-----+
|adsh                |tag                                                                 |ddate   |qtrs|uom|value           |segments                                                                    |coreg|
+--------------------+--------------------------------------------------------------------+--------+----+---+----------------+----------------------------------------------------------------------------+-----+
|0000070145-25-000007|IncreaseDecreaseInAccountsPayableTrade                              |20241231|1   |USD|9703000.0000    |NULL                                                                        |NULL |
|0000351569-25-000006|OtherIntangibleAssetsNet                                            |20241231|0   |USD|70761000.0000   |NULL                              

In [7]:
# Inspect company and filing metadata.
sub.select(
    "adsh",
    "cik",
    "name",
    "sic",
    "form",
    "fy",
    "fp",
    "period"
).show(10, truncate=False)

+--------------------+-------+----------------------------------------+----+----+----+---+--------+
|adsh                |cik    |name                                    |sic |form|fy  |fp |period  |
+--------------------+-------+----------------------------------------+----+----+----+---+--------+
|0000950170-25-071771|1965934|OVERLAND ADVANTAGE                      |NULL|10-Q|2025|Q1 |20250331|
|0000950170-25-070633|2052153|APOLLO ORIGINATION II (UL) CAPITAL TRUST|NULL|10-Q|2025|Q1 |20250331|
|0000816956-25-000011|816956 |CONMED CORP                             |3845|10-Q|2025|Q1 |20250331|
|0001213900-25-037427|1997201|PS INTERNATIONAL GROUP LTD.             |4731|20-F|2024|FY |20241231|
|0001679273-25-000026|1679273|LAMB WESTON HOLDINGS, INC.              |2030|10-Q|2025|Q3 |20250228|
|0001683168-25-003620|1355848|KARTOON STUDIOS, INC.                   |7812|10-Q|2025|Q1 |20250331|
|0001562528-25-000019|1562528|FRANKLIN BSP REALTY TRUST, INC.         |6798|10-Q|2025|Q1 |20250331|


In [8]:
# Count occurrences of each financial tag.
(
    num.groupBy("tag")
    .count()
    .orderBy(F.desc("count"))
    .show(20, truncate=False)
)

+------------------------------------------------------------------------------------------------+------+
|tag                                                                                             |count |
+------------------------------------------------------------------------------------------------+------+
|InvestmentOwnedAtFairValue                                                                      |516461|
|StockholdersEquity                                                                              |500212|
|InvestmentOwnedAtCost                                                                           |444208|
|RevenueFromContractWithCustomerExcludingAssessedTax                                             |400464|
|InvestmentOwnedBalancePrincipalAmount                                                           |317295|
|StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest                          |306187|
|NetIncomeLoss                                

In [9]:
# Identify common filing types.
(
    sub.groupBy("form")
    .count()
    .orderBy(F.desc("count"))
    .show(20)
)

+------+-----+
|  form|count|
+------+-----+
|  10-Q|16754|
|  10-K| 5530|
|  20-F| 1052|
|   6-K|  674|
| S-1/A|  352|
|   S-1|  295|
|10-Q/A|  224|
|10-K/A|  223|
|POS AM|  161|
|  40-F|  139|
| S-4/A|  131|
| F-1/A|   88|
|   F-1|   84|
|20-F/A|   68|
| F-4/A|   58|
|   S-4|   49|
|   8-K|   37|
| 6-K/A|   32|
|  11-K|   30|
| 424B3|   26|
+------+-----+
only showing top 20 rows



### Relevant financial metrics and fiscal years

In [10]:
# Inspect alternative revenue tags and other financial metrics.
financial_tags = [
    "RevenueFromContractWithCustomerExcludingAssessedTax",
    "Revenues",
    "SalesRevenueNet",
    "NetIncomeLoss",
    "Assets",
    "Liabilities",
    "StockholdersEquity"
]

(
    num.filter(F.col("tag").isin(financial_tags))
    .groupBy("tag")
    .count()
    .orderBy(F.desc("count"))
    .show(truncate=False)
)

+---------------------------------------------------+------+
|tag                                                |count |
+---------------------------------------------------+------+
|StockholdersEquity                                 |500212|
|RevenueFromContractWithCustomerExcludingAssessedTax|400464|
|NetIncomeLoss                                      |267704|
|Revenues                                           |184147|
|Assets                                             |127454|
|Liabilities                                        |70889 |
+---------------------------------------------------+------+



In [12]:
# Check which fiscal years are represented in annual filings.
(
    sub.filter(F.col("form") == "10-K")
    .groupBy("fy")
    .count()
    .orderBy(F.desc("count"))
    .show()
)

+----+-----+
|  fy|count|
+----+-----+
|2024| 4824|
|2025|  676|
|2023|   22|
|2022|    8|
+----+-----+



## 4. Filter and join financial records
Keep original annual 10-K filings, US business addresses, and consolidated USD facts. Select only necessary columns before joining. Use the most represented fiscal year (inspect the output); this is an analytical choice, not necessarily 2025.

In [11]:
# Filter annual filings and prune unused columns early.
annual = (sub.filter((F.col("form") == "10-K") & (F.col("countryba") == "US"))
          .select("adsh", "cik", "name", "sic", "fy", "period", "filed")
          .filter(F.col("fy").rlike(r"^[0-9]{4}$")))
years = annual.groupBy("fy").count().orderBy(F.desc("count"), F.desc("fy"))
years.show()
TARGET_FY = years.first()["fy"]  # Override if a different fiscal year is desired.
print("Selected fiscal year:", TARGET_FY)
from pyspark.sql.window import Window
# Keep one original annual filing per company for the selected fiscal year.
w = Window.partitionBy("cik").orderBy(F.desc("period"), F.desc("filed"), F.desc("adsh"))
annual = (annual.filter(F.col("fy") == TARGET_FY)
          .withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn"))

+----+-----+
|  fy|count|
+----+-----+
|2024| 4521|
|2025|  602|
|2023|   17|
|2022|    7|
+----+-----+

Selected fiscal year: 2024


In [12]:
# Keep consolidated, standard US GAAP facts measured in US dollars.
facts = (num.filter(F.col("tag").isin(financial_tags))
         .filter(F.col("uom") == "USD")
         .filter(F.col("version").startswith("us-gaap/"))
         .filter(F.col("segments").isNull() | (F.trim("segments") == ""))
         .filter(F.col("coreg").isNull() | (F.trim("coreg") == ""))
         .select("adsh", "tag", "ddate", "qtrs", "value")
         .withColumn("value", F.col("value").cast("double"))
         .filter(F.col("value").isNotNull()))
# Broadcast the smaller filing table to avoid shuffling the large fact table for the join.
joined = facts.join(F.broadcast(annual), "adsh")
# Match the current reporting period, not comparative prior-year facts.
# SEC fact dates are month-end dates; normalize the filing period accordingly.
joined = joined.filter(F.col("ddate") == F.date_format(F.last_day(F.to_date("period", "yyyyMMdd")), "yyyyMMdd"))
stock_tags = ["Assets", "Liabilities", "StockholdersEquity"]
joined = joined.filter((F.col("tag").isin(stock_tags) & (F.col("qtrs") == "0")) |
                       (~F.col("tag").isin(stock_tags) & (F.col("qtrs") == "4")))
joined.show(10, truncate=False)

+--------------------+---------------------------------------------------+--------+----+----------+-------+-----------------------------+----+----+--------+--------+
|adsh                |tag                                                |ddate   |qtrs|value     |cik    |name                         |sic |fy  |period  |filed   |
+--------------------+---------------------------------------------------+--------+----+----------+-------+-----------------------------+----+----+--------+--------+
|0001462056-25-000015|RevenueFromContractWithCustomerExcludingAssessedTax|20241231|4   |1.27628E8 |1462056|BACKBLAZE, INC.              |7372|2024|20241231|20250311|
|0000950170-25-038803|StockholdersEquity                                 |20241231|0   |6.6327E7  |1841387|CANDEL THERAPEUTICS, INC.    |2836|2024|20241231|20250313|
|0000950170-25-034761|NetIncomeLoss                                      |20241231|4   |-1.3485E8 |1810182|ALX ONCOLOGY HOLDINGS INC    |2834|2024|20241231|20250306|
|000

## 5. Build company-level financial metrics
Exclude conflicting duplicate facts rather than arbitrarily choosing a value. Use a prioritized revenue tag as a fallback, not a sum of alternative revenue definitions. Ratios are descriptive; revenue definitions can differ across industries.

In [13]:
# Collapse identical facts and exclude conflicting values for the same filing and tag.
keys = ["adsh", "cik", "name", "sic", "fy"]
unique_facts = (joined.groupBy(*keys, "tag")
                .agg(F.min("value").alias("lo"), F.max("value").alias("hi"))
                .filter(F.col("lo") == F.col("hi")))
# Pivot financial tags into columns, one row per filing.
wide = unique_facts.groupBy(*keys).pivot("tag", financial_tags).agg(F.first("lo"))
companies = (wide.withColumn("revenue", F.coalesce(*[F.col(t) for t in financial_tags[:3]]))
             .withColumn("profit_margin_pct", F.when(F.col("revenue") > 0, 100 * F.col("NetIncomeLoss") / F.col("revenue")))
             .withColumn("liabilities_to_assets", F.when(F.col("Assets") > 0, F.col("Liabilities") / F.col("Assets")))
             .withColumn("return_on_assets_pct", F.when(F.col("Assets") > 0, 100 * F.col("NetIncomeLoss") / F.col("Assets")))
             .withColumn("industry", F.lpad(F.col("sic"), 4, "0").substr(1, 2))
             .filter(F.col("sic").rlike(r"^[0-9]{1,4}$")))
companies.select("name", "fy", "industry", "revenue", "profit_margin_pct", "liabilities_to_assets").show(10, truncate=False)

+-----------------------------+----+--------+----------+-------------------+---------------------+
|name                         |fy  |industry|revenue   |profit_margin_pct  |liabilities_to_assets|
+-----------------------------+----+--------+----------+-------------------+---------------------+
|MEDICUS PHARMA LTD.          |2024|28      |NULL      |NULL               |0.4448783912101519   |
|SAREPTA THERAPEUTICS, INC.   |2024|28      |1.78796E9 |13.156837960580773 |0.6145154400274729   |
|GREEN THUMB INDUSTRIES INC.  |2024|01      |1.137141E9|6.426907481130308  |0.2948082232169182   |
|ADOBE INC.                   |2024|73      |2.1505E10 |25.854452452917926 |0.5334105193516374   |
|GENELUX CORP                 |2024|28      |8000.0    |-373362.5          |0.2431731766332527   |
|JUPITER NEUROSCIENCES, INC.  |2024|28      |NULL      |NULL               |0.32735683828654316  |
|HYPERFINE, INC.              |2024|38      |1.289E7   |-315.90380139643133|0.16741651245309927  |
|TRAWS PHA

## 6. Industry aggregation and Spark SQL
Use two-digit SIC groups and report the number of non-null observations for each ratio. Require at least 10 observations for rankings.

In [14]:
industry_summary = (companies.groupBy("fy", "industry")
    .agg(F.count("*").alias("companies"), F.count("profit_margin_pct").alias("margin_n"),
         F.count("liabilities_to_assets").alias("leverage_n"),
         F.expr("percentile_approx(profit_margin_pct, 0.5)").alias("median_margin_pct"),
         F.expr("percentile_approx(liabilities_to_assets, 0.5)").alias("median_leverage")))
industry_summary.orderBy(F.desc("companies")).show(20)

+----+--------+---------+--------+----------+-------------------+-------------------+
|  fy|industry|companies|margin_n|leverage_n|  median_margin_pct|    median_leverage|
+----+--------+---------+--------+----------+-------------------+-------------------+
|2024|      28|      711|     373|       647|-110.04326328800988|0.43244648231616417|
|2024|      73|      478|     404|       428|-3.2442930980507407| 0.6095690844962429|
|2024|      67|      402|     168|       390|  4.252387194072306| 0.5634038164002063|
|2024|      60|      331|      29|       331| 28.007607862197382|  0.897586753431332|
|2024|      38|      254|     189|       230| -19.56183586648925| 0.4920445053292793|
|2024|      36|      181|     144|       166|-15.345135505349123|0.49836933015272444|
|2024|      62|      142|      64|       134| 16.728397231792204|0.07282060334014405|
|2024|      49|      132|      85|        71| 11.789887770450076| 0.6908623236280546|
|2024|      13|      115|      83|        91|  4.39322

In [15]:
# Register a temporary SQL view; this does not write data to disk.
companies.createOrReplaceTempView("companies")
# SQL query 1: Which industries have the highest median profit margins?
q1 = spark.sql("""
SELECT industry, COUNT(profit_margin_pct) AS observations,
       percentile_approx(profit_margin_pct, 0.5) AS median_margin_pct
FROM companies GROUP BY industry
HAVING COUNT(profit_margin_pct) >= 10
ORDER BY median_margin_pct DESC LIMIT 10
""")
q1.show(truncate=False)

26/11/24 17:32:24 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+--------+------------+------------------+
|industry|observations|median_margin_pct |
+--------+------------+------------------+
|60      |29          |28.007607862197382|
|62      |64          |16.728397231792204|
|49      |85          |11.789887770450076|
|82      |11          |9.23692343003287  |
|15      |20          |8.724033590589196 |
|63      |95          |8.335917929142312 |
|70      |24          |7.9448234177114   |
|30      |16          |7.092441905861587 |
|14      |10          |6.956800695411012 |
|35      |93          |5.0490113098108855|
+--------+------------+------------------+



In [16]:
# SQL query 2: Which industries have the highest median liabilities-to-assets ratios?
q2 = spark.sql("""
SELECT industry, COUNT(liabilities_to_assets) AS observations,
       percentile_approx(liabilities_to_assets, 0.5) AS median_leverage
FROM companies GROUP BY industry
HAVING COUNT(liabilities_to_assets) >= 10
ORDER BY median_leverage DESC LIMIT 10
""")
q2.show(truncate=False)

+--------+------------+------------------+
|industry|observations|median_leverage   |
+--------+------------+------------------+
|60      |331         |0.897586753431332 |
|70      |26          |0.8657256827247622|
|61      |108         |0.8572447487607432|
|72      |12          |0.837016833648541 |
|47      |12          |0.835783953932173 |
|45      |15          |0.8152961005800374|
|55      |14          |0.8150211379249603|
|63      |102         |0.7967423409172362|
|58      |35          |0.7888107922735427|
|59      |47          |0.7622887214176988|
+--------+------------+------------------+



## 7. Execution plan and optimization
Inspect ReadSchema, PushedFilters, BroadcastHashJoin and Exchange in the actual plan. CSV predicate pushdown does not imply skipping whole files. Aggregations still require shuffles; eight initial shuffle partitions are a local starting point, not a measured optimum. Capture the plan and SQL query details in the Spark UI.

In [17]:
# Show the physical execution plan for the industry aggregation.
industry_summary.explain("formatted")
print("Spark UI:", spark.sparkContext.uiWebUrl)

== Physical Plan ==
AdaptiveSparkPlan (33)
+- ObjectHashAggregate (32)
   +- Exchange (31)
      +- ObjectHashAggregate (30)
         +- Project (29)
            +- Project (28)
               +- HashAggregate (27)
                  +- Exchange (26)
                     +- HashAggregate (25)
                        +- HashAggregate (24)
                           +- HashAggregate (23)
                              +- Project (22)
                                 +- Filter (21)
                                    +- HashAggregate (20)
                                       +- Exchange (19)
                                          +- HashAggregate (18)
                                             +- Project (17)
                                                +- BroadcastHashJoin Inner BuildRight (16)
                                                   :- Project (3)
                                                   :  +- Filter (2)
                                                   :  

## 8. Lazy transformations versus actions
A transformation constructs a plan; an action triggers execution. Compare the Spark UI before and after count.

In [18]:
from time import perf_counter
start = perf_counter()
lazy_demo = companies.filter(F.col("profit_margin_pct") > 0).select("cik", "profit_margin_pct")
print(f"Transformation definition: {perf_counter() - start:.4f} seconds")

Transformation definition: 0.0114 seconds


In [19]:
start = perf_counter()
print("Profitable companies:", lazy_demo.count())
print(f"Action elapsed: {perf_counter() - start:.4f} seconds")

Profitable companies: 1434
Action elapsed: 1.8282 seconds


## 9. Optional caching experiment
Compare the same aggregation before and after materializing a cache. Timings may vary due to JVM warmup and operating-system caching.

In [20]:
def benchmark(df):
    start = perf_counter()
    result = df.agg(F.avg("profit_margin_pct"), F.count("profit_margin_pct")).collect()
    return perf_counter() - start, result
uncached_time, _ = benchmark(companies)
companies.cache()
start = perf_counter()
companies.count()  # Materialize the cache before timing reuse.
print(f"Cache materialization: {perf_counter() - start:.3f}s")
cached_time, _ = benchmark(companies)
print(f"Uncached: {uncached_time:.3f}s; cached: {cached_time:.3f}s")

Cache materialization: 1.826s
Uncached: 1.794s; cached: 0.058s


## 10. Write results to Parquet
The company output is partitioned by fiscal year (one year in this analysis). Avoid excessive small files in the compact industry summary.

In [21]:
OUTPUT = ROOT / "data" / "processed"
companies.coalesce(2).write.mode("overwrite").partitionBy("fy").parquet(str(OUTPUT / "companies"))
industry_summary.coalesce(1).write.mode("overwrite").parquet(str(OUTPUT / "industry_summary"))
print("Saved results to", OUTPUT)
spark.read.parquet(str(OUTPUT / "industry_summary")).show(5)
companies.unpersist()

Saved results to /Users/tienle/Documents/2026/Duke/Fall 2026/Data Engineering | IDS 706/pyspark/data/processed
+----+--------+---------+--------+----------+-------------------+-------------------+
|  fy|industry|companies|margin_n|leverage_n|  median_margin_pct|    median_leverage|
+----+--------+---------+--------+----------+-------------------+-------------------+
|2024|      28|      711|     373|       647|-110.04326328800988|0.43244648231616417|
|2024|      50|       47|      38|        38|  2.722107210500815| 0.6219134702854714|
|2024|      76|        1|       1|         1| -4.489881164698594| 0.9967106072506695|
|2024|      29|       14|      13|        11| 2.4808445673464683| 0.5529787770010706|
|2024|      48|       74|      58|        58| -4.248768472906404| 0.7319412281520719|
+----+--------+---------+--------+----------+-------------------+-------------------+
only showing top 5 rows



DataFrame[adsh: string, cik: string, name: string, sic: string, fy: string, RevenueFromContractWithCustomerExcludingAssessedTax: double, Revenues: double, SalesRevenueNet: double, NetIncomeLoss: double, Assets: double, Liabilities: double, StockholdersEquity: double, revenue: double, profit_margin_pct: double, liabilities_to_assets: double, return_on_assets_pct: double, industry: string]

In [22]:
print("Application ID:", spark.sparkContext.applicationId)
print("Spark UI:", spark.sparkContext.uiWebUrl)

# Trigger a Spark action
spark.range(1000000).count()

Application ID: local-1795559307730
Spark UI: http://10.198.168.68:4041


1000000

## 11. Submission notes
After running all cells, record actual query findings, observed plan operators and timings in README.md. Capture screenshots of the execution plan, successful output and SQL query details at the Spark UI URL. Export this executed notebook to HTML. Keep Spark running until screenshots are captured. No results or optimization claims should be reported without checking the outputs.